# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# diffusion policy import
from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np
import torch
import torch.nn as nn
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler

# Painting imports
import cv2
from style.diffusion_policy_gml.dataset import PushTStateDataset, GmlDataset
from style.diffusion_policy_gml.network import MemorizationModel, ConditionalUnet1D, compute_noise, compute_orig
from style.diffusion_policy_gml.env import PaintingEnv
import style.diffusion_policy_gml.network as network
import style.diffusion_policy_gml.utils as utils
import load_gml

# General
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
import gerry
import svgpathtools

In [ ]:
pred_horizon = 512
obs_horizon = 1  # pad end but don't pad start, to discourage standing still at start
action_horizon = 1
obs_dim = 0  # x/y
action_dim = 5  # dx/dy/penup

num_diffusion_iters = 100

device = torch.device('cuda')

In [ ]:
# dataset_path = "data/gml_000000.zarr"
# dataset_path = "data/gml_003000.zarr"
dataset_path = "data/gml_by_drawing_PRESERVE_ASPECT_CENTERED_003000.zarr"

with gerry.Stopwatch("Loading dataset"):
    dataset = GmlDataset(
        dataset_path=dataset_path,
        sequence_length=pred_horizon,
        pad_before=0,
        pad_after=0,
        # stride=10,
        action_delta=True,
        action_penlift=True,
        normalize=dict(obs=False, action=True),
        # max_drawings=100
    )
print(f'The number of drawings is {len(dataset.episode_ends)}')
print(dataset.indices.shape)

# Inference


In [ ]:
if True:
    # load pretrained weights
    # This is default, training on 3000 drawings
    folder1 = f'runs/Apr04_21-01-28_eagle'
    # # training on 100 drawings
    # folder1 = f'runs/Apr05_16-14-37_eagle'
    # # training with 512-length trajectories
    # folder1 = 'runs/Apr05_16-48-06_eagle'

    ema_noise_pred_net = ConditionalUnet1D(
        input_dim=action_dim,
        global_cond_dim=obs_dim*obs_horizon,
    )
    ema_noise_pred_net.to(device)
    ema_noise_pred_net.load_state_dict(torch.load(f'{folder1}/ema_noise_pred_net.pth'))
    global_cond = None

In [ ]:
# Noise scheduler
noise_scheduler = DDPMScheduler(
    num_train_timesteps=num_diffusion_iters,
    # the choise of beta schedule has big impact on performance
    # we found squared cosine works the best
    beta_schedule='squaredcos_cap_v2',
    # clip output to [-1,1] to improve stability
    clip_sample=True,
    clip_sample_range=5,
    # our network predicts noise (instead of denoised action)
    prediction_type='epsilon'
)

### Sample test on just normal generation, to test everything is working as expected

In [ ]:
B = 15  # num samples
all_obs = {}
all_actions = {}
all_histories = {}

for horizon in tqdm([40, 80, 160, 320, 640, 1280, 2560]):
    # action_n_init = torch.randn((B, pred_horizon, action_dim), device=device)
    action_n_init = torch.randn((B, horizon, action_dim), device=device)
    history = []

    action_n = network.eval(ema_noise_pred_net, noise_scheduler, action_n_init,
                            global_cond=global_cond[[0]] if global_cond is not None else None,
                            log_history=history)

    action_n = action_n.detach().cpu().numpy()
    action = dataset.unnormalize_action(action_n[..., -3:])

    all_actions[horizon] = action
    all_histories[horizon] = history
    all_obs[horizon] = dataset.unnormalize_obs(action_n[..., :-3])

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(20, 3))
for ax, action, obs in zip(axes, all_actions[2560], all_obs[2560]):
    utils.plot_traj(ax, action, x0=obs[0])
    ax.set_aspect('equal')
    ax.set_ylim()

# Now do edits

## First load a drawing into the correct format
We take the following pipeline:  
svg -> GML json -> normalized dataset format

In [ ]:
# First read-in as svg
root = Path('results') / 'gerry10_edit'
infiles = [root / 'gerry00_in.svg', root / 'gerry01_in.svg']

paths, attributes, svg_attributes = svgpathtools.svg2paths2(infiles[0])

In [ ]:
plt.figure(figsize=(8, 3))
txys = []
h = float(svg_attributes['height'])
t0 = 0
for path in paths:
    s = np.concatenate((np.arange(0, path.length(), 5), [path.length()]))
    t = np.array([path.ilength(s_) for s_ in s])
    xy = np.array([path.point(t_) for t_ in t])
    t_ = np.arange(0, len(t)) * 0.05 + t0
    t0 = t_[-1] + 0.05
    txys.append(np.stack([t_, xy.real, h - xy.imag], axis=1))
    plt.plot(xy.real, h - xy.imag, 'k.')
plt.axis('equal')

In [ ]:
w, h = float(svg_attributes['width']), float(svg_attributes['height'])
display(svg_attributes)

In [ ]:
# now write-out as gml json!!!
# with open(root / 'gerry00_in.xml', 'w') as f:
#     f.write(load_gml.txy_to_gml_xml(txys, [0, w, 0, h]))
with open(root / 'gerry00_in.json', 'w') as f:
    f.write(load_gml.txy_to_gml_json(txys, [0, w, 0, h]))

# read back-in as Drawing
drawing = load_gml.Drawing(root / 'gerry00_in.json', scale_behavior='PRESERVE_ASPECT_CENTERED')
drawing

# Finally, convert to dataset normalized format
obs_n, act_n = dataset.create_normalized_from_drawing(drawing)

In [ ]:
# Plot all round-trip stuff to make sure it looks correct
fig, axes = plt.subplots(1, 3, figsize=(10, 2))

for i in range(5):
    axes[0].plot(drawing.strokes[i][:, 1], drawing.strokes[i][:, 2], 'k.', markersize=1)
axes[0].axis('equal');
axes[0].set_title('GML drawing')

axes[1].plot(*dataset.unnormalize_obs(obs_n).T, 'k.', markersize=1)
axes[1].axis('equal')
axes[1].set_title('read-in observation')

pred = np.cumsum(dataset.unnormalize_action(act_n), axis=0)
axes[2].plot(pred[:, 0], pred[:, 1], 'r.', markersize=1)
axes[2].axis('equal');
axes[2].set_title('read-in action, integrated')

## Now do diffusion on the loaded drawing

In [ ]:
def plot_traj(ax, action_n, obs=None, x0=[0, 0], travel_ls='k:', travel_kwargs=dict(), line_ls='.-', **line_kwargs):
    pen_up = action_n[:, 2] > 0.5
    action = dataset.unnormalize_action(action_n)
    
    if obs is None:
        obs = np.concatenate(([[0, 0]], np.cumsum(action[:, :2], axis=0))) + x0

    for i in np.argwhere(pen_up).flatten():
        travel = obs[i] + [[0, 0], action[i, :2].tolist()]
        ax.plot(*travel.T, travel_ls, **travel_kwargs)
    s = -1
    for i in np.argwhere(pen_up).flatten():
        ax.plot(*obs[s + 1:i + 1].T, line_ls, **line_kwargs)
        s = i
    ax.plot(*obs[s + 1:].T, line_ls, **line_kwargs)

def plot_x(ax, x, title):
    x = x.detach().cpu().numpy()
    obs = dataset.unnormalize_obs(x[0, :, :2])
    act = x[0, :, 2:]
    plot_traj(ax, act, x0=obs[0])
    ax.set_title(title)
    ax.set_aspect('equal')

In [ ]:
t_start = 3
repeat = 100
seed = 8675309 + 9

obs_n, act_n = torch.tensor(obs_n), torch.tensor(act_n)
x = torch.concatenate([obs_n, act_n], dim=1).to(device)[None, ...]
x_new = x * 1

torch.manual_seed(seed)
history = [x_new * 1]
for _ in range(repeat):
    x_noisy = network.add_noise(x_new, t_start, noise_scheduler)
    x_new = network.eval_partial(ema_noise_pred_net, noise_scheduler, x_noisy, t_start)
    history.append(x_new * 1)
history = torch.stack(history, dim=0)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True, sharey=True)

plot_x(axes[0], x, 'Original Trajectory')
plot_x(axes[1], x_noisy, 'Noisy Trajectory')
plot_x(axes[2], x_new, 'Denoised Trajectory')
for ax in axes: ax.grid(False)
fig.tight_layout()

if True:
    fig.savefig(root / f'gerry00_out_t{t_start:02d}_r{repeat}_s{seed}.eps')
    np.savez(root / f'gerry00_out_t{t_start:02d}_r{repeat}_s{seed}.npz',
            obs_n=obs_n.detach().cpu().numpy(),
            act_n=act_n.detach().cpu().numpy(),
            x=x.detach().cpu().numpy(),
            x_noisy=x_noisy.detach().cpu().numpy(),
            x_out=x_new.detach().cpu().numpy())


In [ ]:
r, c = 8, 3
fig, axes = plt.subplots(8, 3, figsize=(16, 16), sharex=True, sharey=True)

inds_to_plot = np.linspace(0, len(history) - 1, r * c - 1).astype(int)
for ax, i, xplot in zip(axes.flatten(), inds_to_plot, history[inds_to_plot]):
    plot_x(ax, xplot, f'After {i} iterations')
plot_x(axes[-1][-1], history[-1], 'Denoised Trajectory')
for ax in axes.flatten(): ax.grid(False)
fig.tight_layout()

if True:
    fig.savefig(
        root / f'gerry00_out_t{t_start:02d}_r{repeat}_s{seed}_progression.eps')
    np.savez(root /
             f'gerry00_out_t{t_start:02d}_r{repeat}_s{seed}_progression.npz',
             history=history.detach().cpu().numpy())